# 1. Boolean Matching

In [6]:
import json
import string
from nltk.corpus import stopwords


def load_json_file(file_path):
    """
    Load a JSON file and return its content.

    Args:
        file_path (str): The path to the JSON file.

    Returns:
        dict: The content of the JSON file.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return json.load(file)
    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
        return None
    except json.JSONDecodeError:
        print(f"Error: Failed to decode the JSON file {file_path}.")
        return None


def preprocess_query(query):
    """
    Preprocess the query by removing punctuation and stopwords.

    Args:
        query (str): The user's query.

    Returns:
        list: A list of preprocessed keywords.
    """
    # Remove punctuation
    query = query.translate(str.maketrans('', '', string.punctuation))
    # Convert to lowercase and split into words
    words = query.lower().split()
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    medical_stopwords = {'disease', 'patient', 'treatment', 'condition', 'symptom', 'doctor', 'health', 'may', 'also', 'one', 'use'}
    stop_words.update(medical_stopwords)
    
    filtered_words = [word for word in words if word not in stop_words]
    return filtered_words


def boolean_matching_search(query, keyword_index):
    """
    Perform a boolean matching search based on the query using the keyword index.

    Args:
        query (str): The user's query.
        keyword_index (dict): The inverted index of keywords.

    Returns:
        list: A list of document IDs that match the query.
    """
    query_keywords = preprocess_query(query)
    result_sets = []
    for keyword in query_keywords:
        if keyword in keyword_index:
            result_sets.append(set(keyword_index[keyword]))
    
    if not result_sets:
        return []
    
    # Intersection of all result sets
    final_result = set.intersection(*result_sets)
    return list(final_result)


def retrieve_qa_pairs(result_ids, qa_database):
    """
    Retrieve QA pairs based on the document IDs.

    Args:
        result_ids (list): A list of document IDs.
        qa_database (dict): The QA database.

    Returns:
        list: A list of QA pairs that match the document IDs.
    """
    return [qa for qa in qa_database if qa['id'] in result_ids]


if __name__ == "__main__":
    # Load JSON files
    qa_database = load_json_file('processed_data/qa_database.json')
    keyword_index = load_json_file('processed_data/keyword_index.json')

    if qa_database and keyword_index:
        # Example query
        query = input(f"Please type your query here: ")
        # Perform boolean matching search
        result_ids = boolean_matching_search(query, keyword_index)
        # Retrieve QA pairs
        result_qa_pairs = retrieve_qa_pairs(result_ids, qa_database)

        # Print the results
        print(f"Query: {query}")
        print(f"Number of matching QA pairs: {len(result_qa_pairs)}")
        print()
        n = 1
        for qa in result_qa_pairs:
            print(f"------- No. {n} -------")
            print(f"ID: {qa['id']}, Question: {qa['question']}")
            print(f"Answer: {qa['answer']}")
            print(f"Page URL: {qa['url']}")
            n += 1
            print()

Please type your query here:  why migraine?


Query: why migraine?
Number of matching QA pairs: 12

------- No. 1 -------
ID: healthline_33, Question: everything you need to know about headache disorders
Answer: headaches can occur as a symptom of another health condition such as an infection or as part of a headache disorder headache disorders may cause other symptoms including sensitivity to light or sound and visual changes headaches are widespread and felt by almost everyone experts estimated that 50 to 75 percent of adults had a headache in 2020 often headaches are short term and mild but some can be debilitating and disrupt your daily life there are several kinds of headaches caused by various factors such as our environment the medication we take and other causes many treatment options are available to help manage the pain you can learn more about headaches including migraine headaches and treatments that can help you live a happier and healthier life headache disorders are painful with discomfort felt in the head neck and 

# 2. Semantic Matching

In [5]:
import json
import string
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def load_json_file(file_path):
    """
    Load a JSON file and return its content.

    Args:
        file_path (str): The path to the JSON file.

    Returns:
        dict: The content of the JSON file.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return json.load(file)
    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
        return None
    except json.JSONDecodeError:
        print(f"Error: Failed to decode the JSON file {file_path}.")
        return None


def preprocess_query(query):
    """
    Preprocess the query by removing punctuation and stopwords.

    Args:
        query (str): The user's query.

    Returns:
        list: A list of preprocessed keywords.
    """
    # Remove punctuation
    query = query.translate(str.maketrans('', '', string.punctuation))
    # Convert to lowercase and split into words
    words = query.lower().split()

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    medical_stopwords = {'disease', 'patient', 'treatment', 'condition', 'symptom', 'doctor', 'health', 'may', 'also', 'one', 'use'}
    stop_words.update(medical_stopwords)

    filtered_words = [word for word in words if word not in stop_words]
    return filtered_words


def boolean_matching_search(query, keyword_index):
    """
    Perform a boolean matching search based on the query using the keyword index.

    Args:
        query (str): The user's query.
        keyword_index (dict): The inverted index of keywords.

    Returns:
        list: A list of document IDs that match the query.
    """
    query_keywords = preprocess_query(query)
    result_sets = []
    for keyword in query_keywords:
        if keyword in keyword_index:
            result_sets.append(set(keyword_index[keyword]))

    if not result_sets:
        return []

    # Intersection of all result sets
    final_result = set.intersection(*result_sets)
    return list(final_result)


def retrieve_qa_pairs(result_ids, qa_database):
    """
    Retrieve QA pairs based on the document IDs.

    Args:
        result_ids (list): A list of document IDs.
        qa_database (dict): The QA database.

    Returns:
        list: A list of QA pairs that match the document IDs.
    """
    return [qa for qa in qa_database if qa['id'] in result_ids]


def semantic_matching(query, qa_pairs, max_results=10):
    """
    Perform semantic matching between the query and QA pairs using cosine similarity.

    Args:
        query (str): The user's query.
        qa_pairs (list): A list of QA pairs.

    Returns:
        list: A list of QA pairs sorted by cosine similarity in descending order.
    """
    model = SentenceTransformer('all-mpnet-base-v2')
    query_embedding = model.encode(query)
    qa_embeddings = [model.encode(qa['question']) for qa in qa_pairs]

    similarities = cosine_similarity([query_embedding], qa_embeddings)[0]
    sorted_indices = np.argsort(similarities)[::-1]
    sorted_qa_pairs = [qa_pairs[i] for i in sorted_indices[:max_results]]
    return sorted_qa_pairs


if __name__ == "__main__":
    # Load JSON files
    qa_database = load_json_file('processed_data/qa_database.json')
    keyword_index = load_json_file('processed_data/keyword_index.json')

    if qa_database and keyword_index:
        # Example query
        query = input(f"Please type your query here: ")
        # Perform boolean matching search
        result_ids = boolean_matching_search(query, keyword_index)
        # Retrieve QA pairs
        result_qa_pairs = retrieve_qa_pairs(result_ids, qa_database)

        # Perform semantic matching
        sorted_qa_pairs = semantic_matching(query, result_qa_pairs)

        # Print the results
        print(f"Query: {query}")
        print(f"Number of matching QA pairs: {len(sorted_qa_pairs)}")
        print()
        n = 1
        for qa in sorted_qa_pairs:
            print(f"------- No. {n} -------")
            print(f"ID: {qa['id']}, Question: {qa['question']}")
            print(f"Answer: {qa['answer']}")
            print(f"Page URL: {qa['url']}")
            n += 1
            print()

Please type your query here:  why migraine?


Query: why migraine?
Number of matching QA pairs: 10

------- No. 1 -------
ID: healthline_269, Question: is migraine hereditary tips for prevention and more
Answer: while the exact cause of migraine is unknown it s thought to be caused by a combination of factors including genetics migraine is a chronic neurological disorder that causes intense throbbing headaches these headaches may be accompanied by a variety of other symptoms like nausea light or noise sensitivity or mood changes migraine symptoms may appear hours or even days before the headache begins and may continue after the headache has resolved though there s likely more than one underlying cause of migraine people with migraine have a predisposition to this type of headache along with a sensitivity to triggers factors like genetics or a history of head trauma can predispose someone to migraine while triggers like alcohol or stress can also cause someone to have a migraine episode people can also experience migraine without 

# 3. Keywords Weighting

## 3.1 Define different weighting functions

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi
import spacy
from spacy.matcher import Matcher


def calculate_tfidf_weights(documents):
    """
    Calculate TF-IDF weights for each keyword in the documents.

    Args:
        documents (list): List of documents (strings).

    Returns:
        dict: Keyword to TF-IDF weight mapping.
    """
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(documents)
    feature_names = vectorizer.get_feature_names_out()
    weights = {}
    for col in tfidf_matrix.nonzero()[1]:
        keyword = feature_names[col]
        weights[keyword] = tfidf_matrix[0, col]
    return weights


def calculate_bm25_weights(documents):
    """
    Calculate BM25 weights for each keyword in the documents.

    Args:
        documents (list): List of documents (strings).

    Returns:
        BM25Okapi: Trained BM25 model.
    """
    tokenized_documents = [doc.split() for doc in documents]
    bm25 = BM25Okapi(tokenized_documents)
    return bm25


# 加载医疗领域模型
nlp = spacy.load("en_core_sci_lg")
matcher = Matcher(nlp.vocab)
# 定义实体模式
entity_patterns = [
    [{"LOWER": "diabetes"}],
    [{"LOWER": "heart"}, {"LOWER": "attack"}],
    # 添加更多健康科学实体模式
]
matcher.add("HealthEntities", entity_patterns)

def calculate_entity_weights(query):
    """
    Calculate weights based on health science entities in the query.

    Args:
        query (str): User's query.

    Returns:
        dict: Keyword to weight mapping.
    """
    doc = nlp(query)
    matches = matcher(doc)
    entity_keywords = []
    for match_id, start, end in matches:
        entity = doc[start:end].text
        entity_keywords.append(entity)
    
    weights = {}
    for token in doc:
        if token.text in entity_keywords:
            weights[token.text] = 2.0    # 实体词权重加倍
        elif token.pos_ in ["VERB", "ADJ"]:    # 动词/形容词权重
            weights[token.text] = 1.5
        else:
            weights[token.text] = 1.0
    return weights


def combined_weighting(query, tfidf_weights, bm25, entity_weights):
    """
    Combine multiple weighting strategies into a unified score.

    Args:
        query (str): User's query.
        tfidf_weights (dict): TF-IDF weights.
        bm25 (BM25Okapi): BM25 model.
        entity_weights (dict): Entity-based weights.
        embedding_weights (dict): Embedding-based weights.

    Returns:
        dict: Combined keyword weights.
    """
    combined = {}
    for word in query.split():
        tfidf = tfidf_weights.get(word, 0.0)
        bm25_score = bm25.get_scores([word]) if bm25 else 0.0
        entity = entity_weights.get(word, 1.0)
        # embedding = embedding_weights.get(word, 1.0)
        
        # Linear weights combining 
        combined[word] = 0.4*tfidf + 0.3*bm25_score + 0.3*entity
        # combined[word] = 0.4*tfidf + 0.3*bm25_score + 0.2*entity + 0.1*embedding
    return combined

## 3.2 Implement combined weights to execute boolean matching

In [18]:
def boolean_weighted_matching_search(query, keyword_index, combined_weights):
    """
    Perform a boolean matching search based on the query using the keyword index with combined weights.

    Args:
        query (str): The user's query.
        keyword_index (dict): The inverted index of keywords.
        combined_weights (dict): Combined keyword weights.

    Returns:
        list: A list of document IDs that match the query sorted by weighted score.
    """
    query_keywords = preprocess_query(query)
    score_dict = {}
    for keyword in query_keywords:
        if keyword in keyword_index:
            weight = combined_weights.get(keyword, 1.0)
            # 如果是数组，取其均值
            if isinstance(weight, (np.ndarray, list)):
                weight = np.mean(weight)
            
            for doc_id in keyword_index[keyword]:
                score_dict[doc_id] = score_dict.get(doc_id, 0) + weight
    sorted_doc_ids = sorted(score_dict.keys(), key=lambda x: score_dict[x], reverse=True)
    return sorted_doc_ids


if __name__ == "__main__":
    # Load JSON files
    qa_database = load_json_file('processed_data/qa_database.json')
    keyword_index = load_json_file('processed_data/keyword_index.json')

    if qa_database and keyword_index:
        # Calculate TF-IDF weights
        documents = [qa['question'] for qa in qa_database]
        tfidf_weights = calculate_tfidf_weights(documents)
        # Calculate BM25 weights
        bm25 = calculate_bm25_weights(documents)

        # Example query
        query = input(f"Please type your query here: ")
        # Calculate entity weights
        entity_weights = calculate_entity_weights(query)
        # Calculate combined weights
        combined_weights = combined_weighting(query, tfidf_weights, bm25, entity_weights)

        # Perform boolean matching search with combined weights
        result_ids = boolean_weighted_matching_search(query, keyword_index, combined_weights)
        # Retrieve QA pairs
        result_qa_pairs = retrieve_qa_pairs(result_ids, qa_database)

        # Perform semantic matching
        sorted_qa_pairs = semantic_matching(query, result_qa_pairs)

        # Print the results
        print(f"Query: {query}")
        print(f"Number of matching QA pairs: {len(sorted_qa_pairs)}")
        print()
        n = 1
        for qa in sorted_qa_pairs:
            print(f"------- No. {n} -------")
            print(f"ID: {qa['id']}, Question: {qa['question']}")
            print(f"Answer: {qa['answer']}")
            print(f"Page URL: {qa['url']}")
            n += 1
            print()

Please type your query here:  what is the therapy of migraine?


Query: what is the therapy of migraine?
Number of matching QA pairs: 10

------- No. 1 -------
ID: healthline_269, Question: is migraine hereditary tips for prevention and more
Answer: while the exact cause of migraine is unknown it s thought to be caused by a combination of factors including genetics migraine is a chronic neurological disorder that causes intense throbbing headaches these headaches may be accompanied by a variety of other symptoms like nausea light or noise sensitivity or mood changes migraine symptoms may appear hours or even days before the headache begins and may continue after the headache has resolved though there s likely more than one underlying cause of migraine people with migraine have a predisposition to this type of headache along with a sensitivity to triggers factors like genetics or a history of head trauma can predispose someone to migraine while triggers like alcohol or stress can also cause someone to have a migraine episode people can also experienc